In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

In [2]:
Bell_2D = xr.load_dataset(
    "/mnt/c/Users/banga/OneDrive - University of Southampton/Documents/PhD/CUSTARD analysis/data/glider/processed/Bellamite/Bellamite_2D.nc"
)
Bell_2D

<xarray.Dataset> Size: 93MB
Dimensions:               (PROFILE_NUMBER: 586, DEPTH_BIN: 548)
Coordinates:
  * PROFILE_NUMBER        (PROFILE_NUMBER) float64 5kB 24.0 26.0 ... 895.0 896.0
    median_TIME           (PROFILE_NUMBER) datetime64[ns] 5kB 2019-12-06T12:4...
  * DEPTH_BIN             (DEPTH_BIN) float64 4kB -1.0 0.0 1.0 ... 545.0 546.0
Data variables: (12/60)
    BBP700                (PROFILE_NUMBER, DEPTH_BIN) float64 3MB nan ... nan
    CDOM                  (PROFILE_NUMBER, DEPTH_BIN) float64 3MB nan ... nan
    CHLA                  (PROFILE_NUMBER, DEPTH_BIN) float64 3MB nan ... nan
    TIME                  (PROFILE_NUMBER, DEPTH_BIN) datetime64[ns] 3MB NaT ...
    MOLAR_DOXY            (PROFILE_NUMBER, DEPTH_BIN) float64 3MB nan ... nan
    CNDC                  (PROFILE_NUMBER, DEPTH_BIN) float64 3MB nan ... nan
    ...                    ...
    N2                    (PROFILE_NUMBER, DEPTH_BIN) float64 3MB nan ... nan
    MLD                   (PROFILE_NUMBER) float64 5kB nan nan nan ... nan nan
    MLD_error             (PROFILE_NUMBER) float64 5kB nan nan nan ... nan nan
    AML                   (PROFILE_NUMBER) float64 5kB nan nan nan ... nan nan
    AML_error             (PROFILE_NUMBER) float64 5kB nan nan nan ... nan nan
    turb_AML              (PROFILE_NUMBER) float64 5kB nan nan nan ... nan nan

In [3]:
def to_pylag_ds(dataset):

    time = dataset["median_TIME"].values
    zi = np.tile(dataset["DEPTH_BIN"].values * -1, (len(time), 1))[
        :, :, np.newaxis, np.newaxis
    ]
    z = zi[:, :-1] - 0.5

    pylag_ds = xr.Dataset(
        {
            "nuh": (
                ("time", "z", "lat", "lon"),
                dataset["dissipation"].values[:, :-1, np.newaxis, np.newaxis],
            ),
            "zeta": (("time", "lat", "lon"), np.full((len(time), 1, 1), np.max(zi))),
        },
        coords={
            "time": (("time"), time),
            "zi": (("time", "zi", "lat", "lon"), zi),
            "z": (("time", "z", "lat", "lon"), z),
            "lat": (("lat"), [-54]),
            "lon": (("lon"), [-89]),
        },
    )
    return pylag_ds


def convert_grid(dataset, dz, dt=3600, fpath=None):

    time = dataset["time"].values
    K = dataset["nuh"].squeeze().values
    zi = dataset["zi"].squeeze().values[0, :]
    z = dataset["z"].squeeze().values[0, :]

    new_time = np.arange(
        np.min(time).astype("datetime64[h]"),
        np.max(time).astype("datetime64[h]"),
        np.timedelta64(dt, "s"),
    )
    new_zi = np.arange(np.min(zi), np.max(zi) + dz, dz)

    temp_K = xr.DataArray(K, coords={"time": time, "z": z})
    new_K = temp_K.interp(time=new_time, z=new_zi, method="linear")

    new_zi = np.tile(new_zi, (len(new_time), 1))[:, :, np.newaxis, np.newaxis]
    new_z = new_zi[:, :-1] + dz / 2

    interp_ds = xr.Dataset(
        {
            "nuh": (
                ("time", "zi", "lat", "lon"),
                new_K.values[:, :, np.newaxis, np.newaxis],
            ),
            "zeta": (
                ("time", "lat", "lon"),
                np.full((len(new_time), 1, 1), np.max(new_zi)),
            ),
        },
        coords={
            "time": (("time"), new_time),
            "zi": (("time", "zi", "lat", "lon"), new_zi),
            "z": (("time", "z", "lat", "lon"), new_z),
            "lat": (("lat"), [-54]),
            "lon": (("lon"), [-89]),
        },
    )

    interp_ds = interp_ds.interpolate_na(dim="zi")
    interp_ds = interp_ds.interpolate_na(dim="time", fill_value="extrapolate")

    if fpath is not None:
        interp_ds.to_netcdf(fpath)

    return interp_ds


def gen_particle_positions(dataset, N, order=1, plot=False, fpath=None):

    from scipy.interpolate import CubicSpline
    import matplotlib.pyplot as plt

    initial_bbp = dataset.BBP700_ADJUSTED.isel({"PROFILE_NUMBER": slice(0, 20)})
    initial_bbp = initial_bbp.mean(dim="PROFILE_NUMBER") ** order
    norm_bbp = initial_bbp.values / np.sum(initial_bbp.values)
    depths = initial_bbp.DEPTH_BIN.values
    pdf = CubicSpline(depths, norm_bbp)

    zi = []
    xi, yi = [0] * N, [0] * N

    max_depth = np.max(depths)
    min_dz = max_depth / (1000 * N)
    prior_z = 0
    while len(zi) < N:
        current_z = prior_z
        while np.sum(pdf.integrate(prior_z, current_z)) < (1 / N):
            current_z += min_dz
        zi.append((prior_z + current_z) / 2)
        prior_z = current_z
        if prior_z > max_depth:
            break

    if plot:
        fig, ax = plt.subplots(figsize=(18, 5))
        x = np.linspace(0, np.max(depths), 1000)
        ax.plot(x, pdf(x))
        ax.scatter(zi, np.full_like(zi, np.mean(pdf(x))), c="k", marker=".")

    if fpath is not None:
        with open(fpath, "w") as fh:
            fh.write(f"{len(zi)}\n")
            for x, y, z in zip(xi, yi, zi):
                fh.write(f"0 {x} {y} {-z}\n")

    return zi, pdf


class Simulator:

    def __init__(self, ds, N_particles, dz, dt, sim_dir, order=1):

        from pathlib import Path
        import configparser

        self.sim_path = Path("sim_runs").joinpath(sim_dir)
        self.in_dir = self.sim_path.joinpath("input")
        self.out_dir = self.sim_path.joinpath("output")
        for p in [self.in_dir, self.out_dir]:
            p.mkdir(parents=True, exist_ok=True)

        ds_in = to_pylag_ds(ds)
        ds_in = convert_grid(ds_in, dz, dt, fpath=self.in_dir.joinpath("physics.nc"))
        _, _ = gen_particle_positions(
            ds, N_particles, order, fpath=self.in_dir.joinpath("init_positions.dat")
        )

        config = configparser.ConfigParser()
        config.read("./template.cfg")

        start = np.nanmin(ds.TIME.values).astype("datetime64[s]")
        end = np.nanmax(ds.TIME.values).astype("datetime64[D]").astype("datetime64[s]")

        config["SIMULATION"]["start_datetime"] = str(start).replace("T", " ")
        config["SIMULATION"]["end_datetime"] = str(end).replace("T", " ")
        config["SIMULATION"]["duration_in_days"] = str(
            end.astype("datetime64[D]") - start.astype("datetime64[D]")
        ).split(" ")[0]

        with self.in_dir.joinpath("config.cfg").open("w") as f:
            config.write(f)

    def run(self):

        import subprocess
        import os

        try:
            result = subprocess.run(
                [
                    "python",
                    "-m",
                    "pylag.main",
                    "-c",
                    "./input/config.cfg",
                ],
                cwd=self.sim_path,
                capture_output=True,
                timeout=600,
            )
            if result.returncode != 0:
                print(f"  PyLag run failed:\n{result.stderr.decode()}")
                return None

            output_file = os.path.join(self.out_dir, "sim_result_1.nc")
            if os.path.exists(output_file):
                return xr.open_dataset(output_file)
            else:
                print("  PyLag produced no output file.")
                return None

        except subprocess.TimeoutExpired:
            print("  PyLag simulation timed out.")
            return None
        except Exception as exc:
            print(f"  Error running PyLag: {exc}")
            return None

In [25]:
run_input = Bell_2D.sel(DEPTH_BIN=slice(0, 200)).copy()

def adjustment_func(a, x):
    return (1 / a) * (np.exp(x) - 1) ** 2
a = np.nanquantile(run_input["dissipation"], 0.01)
run_input["dissipation"] = adjustment_func(a, run_input.dissipation)

In [26]:
Bell_sim = Simulator(run_input, 1000, 1, 1800, "Bell")
result = Bell_sim.run()

In [27]:
start_pos = result.z.isel({"time": 0})
end_pos = result.z.isel({"time": -1})
travel_distance = np.abs(end_pos - start_pos).values

idxs = np.argsort(travel_distance)[-10:]
far_parts = result.z.isel({"particles": idxs})

import matplotlib
matplotlib.use("tkagg")
fig, ax = plt.subplots(figsize=(18, 8))
far_parts.plot.line(
    x="time",
    c="b",
    alpha=0.2,
    add_legend=False,
    ax=ax,
)
ax.set(
    title=f"Mean:{travel_distance.mean(): .2f} m | Median: {np.median(travel_distance): .2f} m | Max: {travel_distance.max(): .2f} m",
)
plt.show(block=True)

In [28]:
matplotlib.use("tkagg")
fig, ax = plt.subplots(figsize=(18, 8))
result.z.plot.line(
    x="time",
    c="b",
    alpha=0.1,
    add_legend=False,
    ax=ax,
)
plt.show(block=True)